# Task 2 – Amplitude Scaling

For a subset of ≥500 samples, randomly multiply the amplitude by a factor drawn
uniformly from **{25, 1, 0.04}** and compare system performance with Task 1 (baseline).

- Factor **25** → very loud, will hard-clip to ±1.0 (introduces distortion)
- Factor **1** → unchanged (control group)
- Factor **0.04** → very quiet (~−28 dB relative to original)

**Prerequisites:** Task 1 completed (`results/task1_results.json` exists).

## 0. Setup

In [ ]:
import sys, json, random
from pathlib import Path

SRC = Path("../src").resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from config import cfg
from embeddings import load_model, load_audio
from database import get_collection, list_enrolled
from threshold import compute_eer, compute_metrics_at_threshold, _load_enrolled_embeddings_bulk

RESULTS_DIR = cfg.paths.results_dir
TEST_DIR    = cfg.paths.test_dir
RANDOM_SEED = cfg.dataset.random_seed

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Results → {RESULTS_DIR}")

## 1. Load Task 1 baseline results

In [ ]:
task1_path = RESULTS_DIR / "task1_results.json"
if not task1_path.exists():
    raise FileNotFoundError("Run task1_baseline.ipynb first to generate task1_results.json")

with open(task1_path) as f:
    task1 = json.load(f)

print("Task 1 baseline:")
for k, v in task1.items():
    print(f"  {k:<30} {v}")

## 2. Load model and database

In [ ]:
print("Loading ECAPA-TDNN model...")
model = load_model()

collection = get_collection()
enrolled_ids = set(list_enrolled(collection))
print(f"Enrolled speakers: {len(enrolled_ids)}")

if not enrolled_ids:
    raise RuntimeError("No speakers enrolled. Run src/enroll.py first.")

## 3. Build trial list

Collect genuine + impostor pairs from `data/test/`, same logic as Task 1.  
Each test file gets a randomly assigned amplitude factor from {25, 1, 0.04}.

In [ ]:
AMPLITUDE_FACTORS = [25.0, 1.0, 0.04]
MAX_GENUINE_PER_SPEAKER = 50
N_IMPOSTORS_PER_GENUINE = 1

rng = random.Random(RANDOM_SEED)

# Collect test files per enrolled speaker
speaker_files: dict[str, list[Path]] = {}
for spk_dir in sorted(TEST_DIR.iterdir()):
    if spk_dir.is_dir() and spk_dir.name in enrolled_ids:
        wavs = sorted(spk_dir.rglob("*.wav"))
        if wavs:
            speaker_files[spk_dir.name] = wavs

speaker_ids = sorted(speaker_files.keys())
print(f"Speakers with test files: {len(speaker_ids)}")

# Build trial list: (test_path, enrolled_speaker_id, is_genuine, amplitude_factor)
Trial = tuple  # (path, enrolled_id, is_genuine, factor)
trials: list[Trial] = []

for spk_id in speaker_ids:
    files = speaker_files[spk_id][:]
    rng.shuffle(files)
    genuine_files = files[:MAX_GENUINE_PER_SPEAKER]

    for wav in genuine_files:
        factor = rng.choice(AMPLITUDE_FACTORS)
        trials.append((wav, spk_id, True, factor))

    # Impostor trials: other speakers' files vs. this profile
    other_ids = [s for s in speaker_ids if s != spk_id]
    imp_ids = rng.sample(other_ids, min(N_IMPOSTORS_PER_GENUINE * len(genuine_files), len(other_ids)))
    for imp_id in imp_ids:
        imp_wav = rng.choice(speaker_files[imp_id])
        factor = rng.choice(AMPLITUDE_FACTORS)
        trials.append((imp_wav, spk_id, False, factor))

n_genuine  = sum(1 for t in trials if t[2])
n_impostor = sum(1 for t in trials if not t[2])
print(f"Genuine trials:  {n_genuine}")
print(f"Impostor trials: {n_impostor}")
print(f"Total:           {len(trials)}")
assert len(trials) >= 500, "Need ≥500 trials. Add more speakers or increase MAX_GENUINE_PER_SPEAKER."

# Factor distribution
from collections import Counter
factor_counts = Counter(t[3] for t in trials)
print("\nFactor distribution:")
for f, c in sorted(factor_counts.items()):
    print(f"  ×{f:<6} → {c} trials ({c/len(trials)*100:.1f}%)")

## 4. Compute scaled embeddings and scores

Each waveform is scaled by its assigned factor **before** passing to the model.  
Factor 25 will clip the waveform to ±1.0 (hard clipping).

In [ ]:
def get_scaled_embedding(model, audio_path: Path, factor: float) -> np.ndarray:
    """Load audio, scale amplitude by factor (with clipping), extract embedding."""
    waveform = load_audio(audio_path)          # (1, T) float32 tensor
    waveform = waveform * factor
    waveform = waveform.clamp(-1.0, 1.0)       # hard clip – mirrors real-world ADC behaviour

    with torch.no_grad():
        emb = model.encode_batch(waveform.squeeze(0).unsqueeze(0))

    emb = emb.squeeze().cpu().numpy().astype(np.float32)
    norm = np.linalg.norm(emb)
    return emb / norm if norm > 0 else emb


# Bulk-load all enrolled embeddings in a single call to avoid chromadb per-ID get bug
enrolled_cache: dict[str, np.ndarray] = _load_enrolled_embeddings_bulk(collection)
print(f"Loaded {len(enrolled_cache)} enrolled embeddings.")

print(f"Computing {len(trials)} trial scores...")

results_by_factor: dict[float, dict] = {
    f: {"genuine": [], "impostor": []} for f in AMPLITUDE_FACTORS
}

for i, (wav_path, enrolled_id, is_genuine, factor) in enumerate(trials):
    if enrolled_id not in enrolled_cache:
        print(f"  [warn] {enrolled_id}: no enrolled embedding, skipping trial")
        continue

    query_emb    = get_scaled_embedding(model, wav_path, factor)
    enrolled_emb = enrolled_cache[enrolled_id]
    score = float(np.dot(query_emb, enrolled_emb))

    bucket = "genuine" if is_genuine else "impostor"
    results_by_factor[factor][bucket].append(score)

    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(trials)}...")

print("Done.")

# Aggregate all factors
all_genuine  = np.array([s for f in AMPLITUDE_FACTORS for s in results_by_factor[f]["genuine"]])
all_impostor = np.array([s for f in AMPLITUDE_FACTORS for s in results_by_factor[f]["impostor"]])

## 5. Metrics – per factor and combined

In [ ]:
eer_threshold = task1["eer_threshold"]   # reuse threshold found in Task 1

rows = []
print(f"{'Factor':<10} {'N gen':>6} {'N imp':>6} {'EER%':>8} {'FAR%':>8} {'FRR%':>8} {'Acc%':>8}")
print("-" * 62)

for factor in AMPLITUDE_FACTORS:
    gen  = np.array(results_by_factor[factor]["genuine"])
    imp  = np.array(results_by_factor[factor]["impostor"])
    if len(gen) == 0 or len(imp) == 0:
        continue

    eer, eer_thr = compute_eer(gen, imp)
    m = compute_metrics_at_threshold(gen, imp, eer_threshold)
    row = {"factor": factor, "n_genuine": len(gen), "n_impostor": len(imp),
           "eer_pct": round(eer*100, 2), "far_pct": round(m["far"]*100, 2),
           "frr_pct": round(m["frr"]*100, 2), "accuracy_pct": round(m["accuracy"]*100, 2)}
    rows.append(row)
    print(f"  ×{factor:<8} {len(gen):>6} {len(imp):>6} {eer*100:>8.2f} {m['far']*100:>8.2f} {m['frr']*100:>8.2f} {m['accuracy']*100:>8.2f}")

# Combined
eer_all, _ = compute_eer(all_genuine, all_impostor)
m_all = compute_metrics_at_threshold(all_genuine, all_impostor, eer_threshold)
row_all = {"factor": "ALL", "n_genuine": len(all_genuine), "n_impostor": len(all_impostor),
           "eer_pct": round(eer_all*100, 2), "far_pct": round(m_all["far"]*100, 2),
           "frr_pct": round(m_all["frr"]*100, 2), "accuracy_pct": round(m_all["accuracy"]*100, 2)}
rows.append(row_all)

print("-" * 62)
print(f"  {'ALL':<9} {len(all_genuine):>6} {len(all_impostor):>6} "
      f"{eer_all*100:>8.2f} {m_all['far']*100:>8.2f} {m_all['frr']*100:>8.2f} {m_all['accuracy']*100:>8.2f}")
print(f"\n  [Task 1 baseline EER: {task1['eer_pct']:.2f}%  Accuracy: {task1['accuracy_at_eer_pct']:.2f}%]")

## 6. Score distributions – per factor

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
factor_labels = {25.0: "×25  (loud / clipping)", 1.0: "×1  (unchanged)", 0.04: "×0.04  (quiet)"}

for ax, factor in zip(axes, AMPLITUDE_FACTORS):
    gen = np.array(results_by_factor[factor]["genuine"])
    imp = np.array(results_by_factor[factor]["impostor"])

    ax.hist(gen, bins=60, alpha=0.65, color="steelblue", density=True, label="Genuine")
    ax.hist(imp, bins=60, alpha=0.65, color="tomato",    density=True, label="Impostor")
    ax.axvline(eer_threshold, color="black", linestyle="--", linewidth=1.2,
               label=f"thr={eer_threshold:.3f}")

    eer_f, _ = compute_eer(gen, imp)
    m_f = compute_metrics_at_threshold(gen, imp, eer_threshold)
    ax.set_title(f"{factor_labels[factor]}\nEER={eer_f*100:.2f}%  Acc={m_f['accuracy']*100:.1f}%",
                 fontsize=10)
    ax.set_xlabel("Cosine similarity")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Density")
fig.suptitle("Task 2 – Score distributions by amplitude factor", fontsize=13, fontweight="bold")
fig.tight_layout()

out = RESULTS_DIR / "task2_score_distributions.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 7. EER comparison with Task 1

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels      = [f"×{f}" for f in AMPLITUDE_FACTORS] + ["ALL", "Task 1\n(baseline)"]
eer_values  = [next(r["eer_pct"] for r in rows if r["factor"] == f) for f in AMPLITUDE_FACTORS]
eer_values += [row_all["eer_pct"], task1["eer_pct"]]
acc_values  = [next(r["accuracy_pct"] for r in rows if r["factor"] == f) for f in AMPLITUDE_FACTORS]
acc_values += [row_all["accuracy_pct"], task1["accuracy_at_eer_pct"]]

colors = ["steelblue", "mediumseagreen", "steelblue", "slategray", "coral"]

ax = axes[0]
bars = ax.bar(labels, eer_values, color=colors, edgecolor="white", linewidth=0.5)
ax.bar_label(bars, fmt="%.2f%%", fontsize=9)
ax.set_ylabel("EER (%)")
ax.set_title("EER by amplitude factor vs. Task 1")
ax.set_ylim(0, max(eer_values) * 1.3)

ax = axes[1]
bars = ax.bar(labels, acc_values, color=colors, edgecolor="white", linewidth=0.5)
ax.bar_label(bars, fmt="%.2f%%", fontsize=9)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by amplitude factor vs. Task 1")
ax.set_ylim(max(0, min(acc_values) - 5), 100)

fig.suptitle("Task 2 – Amplitude Scaling: Impact on System Performance", fontsize=13, fontweight="bold")
fig.tight_layout()

out = RESULTS_DIR / "task2_comparison.png"
fig.savefig(out, dpi=150)
plt.show()
print(f"Saved: {out}")

## 8. Save results

In [ ]:
summary = {
    "task": 2,
    "description": "Amplitude scaling ×{25, 1, 0.04} applied randomly with uniform probability",
    "eer_threshold_used": eer_threshold,
    "per_factor": [
        {"factor": r["factor"], "n_genuine": r["n_genuine"], "n_impostor": r["n_impostor"],
         "eer_pct": r["eer_pct"], "far_pct": r["far_pct"],
         "frr_pct": r["frr_pct"], "accuracy_pct": r["accuracy_pct"]}
        for r in rows
    ],
    "task1_eer_pct":      task1["eer_pct"],
    "task1_accuracy_pct": task1["accuracy_at_eer_pct"],
}

out_json = RESULTS_DIR / "task2_results.json"
with open(out_json, "w") as f:
    json.dump(summary, f, indent=2)

print("\n═" * 24)
print("  TASK 2 – AMPLITUDE SCALING RESULTS")
print("═" * 24)
print(f"  {'Factor':<12} {'EER%':>8} {'FAR%':>8} {'FRR%':>8} {'Acc%':>8}")
print("  " + "-" * 48)
for r in rows:
    print(f"  ×{str(r['factor']):<11} {r['eer_pct']:>8.2f} {r['far_pct']:>8.2f} "
          f"{r['frr_pct']:>8.2f} {r['accuracy_pct']:>8.2f}")
print("  " + "-" * 48)
print(f"  {'Baseline (T1)':<12} {task1['eer_pct']:>8.2f} {task1['far_at_eer_pct']:>8.2f} "
      f"{task1['frr_at_eer_pct']:>8.2f} {task1['accuracy_at_eer_pct']:>8.2f}")
print("═" * 24)
print(f"\nJSON saved: {out_json}")